In [ ]:
import sys
import importlib
from pathlib import Path

import numpy as np

# Find the repository root robustly from the notebook working directory
cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "utilities" / "functions.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing utilities/functions.py")

sys.path.append(str(repo_root))

import utilities.functions as functions

importlib.reload(functions)

IN_start_index, IN_end_index = 1, 263
PR_start_index, PR_end_index = 1, 99
RT_start_index, RT_end_index = 39, 226

data_root = repo_root / "ms0_5"

IN_all_seq = functions.read_seq(str(data_root / "IN" / "data" / "in.reduce4.seq"))
PR_all_seq = functions.read_seq(str(data_root / "PR" / "data" / "pr.exper.reduce4.seq"))
RT_all_seq = functions.read_seq(str(data_root / "RT" / "data" / "rt.reduce4.seq"))

with open(str(data_root / "IN" / "data" / "in.consensus.reduce4.seq")) as f:
    IN_consensus_seq = f.read().strip()
with open(str(data_root / "PR" / "data" / "pr.consensus.reduce4.seq")) as f:
    PR_consensus_seq = f.read().strip()
with open(str(data_root / "RT" / "data" / "rt.consensus.reduce4.seq")) as f:
    RT_consensus_seq = f.read().strip()

IN_redux = functions.get_redu_dict(str(data_root / "IN" / "data" / "in.reduce4.redux"), 1)
PR_redux = functions.get_redu_dict(str(data_root / "PR" / "data" / "pr.reduce4.redux"), 0)
RT_redux = functions.get_redu_dict(str(data_root / "RT" / "data" / "rt.reduce4.redux"), 0)


def build_J_matrix(j_file, min_position, max_position):
    """Dense coupling tensor, shape (L, L, 4, 5), indexed [p1, p2, aa_at_p1, aa_at_p2].

    Same content as functions.load_J_dict but as an array, so couplings can be gathered
    with numpy. The 5th slot on the last axis is an all-zero column standing in for
    out-of-alphabet characters.
    """
    J = np.load(j_file).astype(np.float32)
    L = max_position - min_position + 1
    Jm = np.zeros((L, L, 4, 5), dtype=np.float32)
    iu0, iu1 = np.triu_indices(L, 1)
    blocks = J.reshape(-1, 4, 4)
    assert blocks.shape[0] == iu0.size, "J.npy row count does not match the position range"
    Jm[iu0, iu1, :, :4] = blocks
    Jm[iu1, iu0, :, :4] = blocks.transpose(0, 2, 1)
    return Jm


IN_J = build_J_matrix(str(data_root / "IN" / "data" / "J.npy"), IN_start_index, IN_end_index)
PR_J = build_J_matrix(str(data_root / "PR" / "data" / "J_PR.npy"), PR_start_index, PR_end_index)
RT_J = build_J_matrix(str(data_root / "RT" / "data" / "J_RT.npy"), RT_start_index, RT_end_index)

with open(str(data_root / "IN" / "data" / "in.weights.txt")) as f:
    IN_weights = np.array([float(l.strip()) for l in f if l.strip()])
with open(str(data_root / "PR" / "data" / "pr.exper.weights.txt")) as f:
    PR_weights = np.array([float(l.strip()) for l in f if l.strip()])
with open(str(data_root / "RT" / "data" / "rt.weights.txt")) as f:
    RT_weights = np.array([float(l.strip()) for l in f if l.strip()])

print("alignments:", len(IN_all_seq), "IN /", len(PR_all_seq), "PR /", len(RT_all_seq), "RT")

In [ ]:
# ---------------------------------------------------------------------------
# The same energy machinery as DMC_sebset_freq2.7.1_c_overlap_flag.ipynb, copied here so
# this notebook stands alone. Nothing about the definitions changes:
#
#   S(p, x) = sum_{o not in {p1, p2}} J[p, o, x, seq[o]]        background, pair excluded
#   T(p, x) = sum_{o != p}            J[p, o, x, seq[o]]        background, full
#   M[a1, a2] = S(p1, a1) + S(p2, a2) + J[p1, p2, a1, a2]
#   dE1 / dE2 / dE double = M[wt1, wt2] - M[mt1, wt2] / M[wt1, mt2] / M[mt1, mt2]
#
# "dE double" is what the figures call the double mutation's dE; the code keeps the
# shorter variable name de12 that it shares with the DMC notebooks.
#   p(DMC) = softmax over -M at (mt1, mt2)
#
# Gain of fitness is the position-local test -- dE double beats the wild type and both of the
# pair's single mutants. The contender requirement (also beating every double mutation in
# conflict with the pair) is off by default and lives behind NON_OVERLAPPING in the next cell;
# it only ever changes the four categories, which only figure G uses.
# ---------------------------------------------------------------------------

_AA_CODE = np.full(256, 4, dtype=np.uint8)  # anything outside ABCD -> the zero-coupling slot
for _i, _c in enumerate("ABCD"):
    _AA_CODE[ord(_c)] = _i


def encode_seqs(seq_list, min_pos, max_pos):
    """One-hot encode an alignment as (N, L*5) float32, column order (position, aa)."""
    L = max_pos - min_pos + 1
    N = len(seq_list)
    raw = np.frombuffer("".join(seq_list).encode(), dtype=np.uint8)
    assert raw.size == N * L, "all sequences must span exactly min_pos..max_pos"
    codes = _AA_CODE[raw.reshape(N, L)]
    onehot = np.zeros((N * L, 5), dtype=np.float32)
    onehot[np.arange(N * L), codes.ravel()] = 1.0
    return codes, onehot.reshape(N, L * 5)


def _site_energies(onehot, Jm, p, excluded):
    """S(p, x) for x in ABCD, for every sequence -> (N, 4)."""
    A = Jm[p].copy()
    A[list(excluded)] = 0.0
    return np.asarray(onehot @ A.transpose(0, 2, 1).reshape(-1, 4), dtype=np.float64)


def pair_energies(onehot, Jm, p1i, p2i, wt1, mt1, wt2, mt2):
    """dE1, dE2, dE double and p(DMC) of one pair, per sequence."""
    S1 = _site_energies(onehot, Jm, p1i, (p1i, p2i))
    S2 = _site_energies(onehot, Jm, p2i, (p1i, p2i))
    J12 = Jm[p1i, p2i, :, :4].astype(np.float64)

    M = S1[:, :, None] + S2[:, None, :] + J12[None, :, :]
    base = M[:, wt1, wt2]
    de1 = base - M[:, mt1, wt2]
    de2 = base - M[:, wt1, mt2]
    de12 = base - M[:, mt1, mt2]

    Z = -M.reshape(S1.shape[0], 16)
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return de1, de2, de12, E[:, mt1 * 4 + mt2] / E.sum(axis=1)


def background_energies(onehot, Jm):
    """T[n, p, x] = sum_{o != p} J[p, o, x, seq[o]], shape (N, L, 4)."""
    L = Jm.shape[0]
    W = np.asarray(Jm, dtype=np.float64).transpose(0, 2, 1, 3).reshape(L * 4, L * 5)
    return (np.asarray(onehot, dtype=np.float64) @ W.T).reshape(-1, L, 4)


def de12_from_T(T, Jm, codes, p, q, a, b, u_p, u_q):
    """dE double of one double mutation (p: u_p -> a, q: u_q -> b), per sequence."""
    sp, sq = codes[:, p], codes[:, q]
    Jpq = np.asarray(Jm[p, q], dtype=np.float64)
    Jqp = np.asarray(Jm[q, p], dtype=np.float64)
    return ((T[:, p, u_p] - T[:, p, a]) + (T[:, q, u_q] - T[:, q, b])
            - (Jpq[u_p, sq] - Jpq[a, sq]) - (Jqp[u_q, sp] - Jqp[b, sp])
            + (Jpq[u_p, u_q] - Jpq[a, b]))


def best_competing_dmc(T, Jm, codes, p, u_p, m_p, u, chunk=2048):
    """Strongest double mutation in conflict with a pair that mutates p to m_p, per sequence."""
    N, L, _ = T.shape
    qs = np.arange(L)
    aa = np.arange(4)
    Jp = np.asarray(Jm[p], dtype=np.float64)
    Jq = np.asarray(Jm[:, p], dtype=np.float64)
    direct = Jp[qs, u_p, u][:, None, None] - Jp[:, :, :4]

    invalid = np.zeros((L, 4, 4), dtype=bool)
    invalid[p] = True             # q must be a different position
    invalid[:, u_p, :] = True     # a must actually be a mutation at p
    invalid[:, m_p, :] = True     # a == m_p agrees with the pair at p -> not a rival
    invalid[qs, :, u] = True      # b must actually be a mutation at q
    invalid = invalid.reshape(L * 16)

    best = np.empty(N)
    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        cb, Ts = codes[s:e], T[s:e]
        sp = cb[:, p]
        dp = Ts[:, p, u_p][:, None] - Ts[:, p, :]
        dq = np.take_along_axis(Ts, u[None, :, None], axis=2) - Ts
        cA = (Jp[qs[None, :], u_p, cb][:, :, None]
              - Jp[qs[None, :, None], aa[None, None, :], cb[:, :, None]])
        cB = (Jq[qs, u, sp[:, None]][:, :, None]
              - Jq[qs[None, :, None], aa[None, None, :], sp[:, None, None]])
        dE = (dp[:, None, :, None] + dq[:, :, None, :]
              - cA[:, :, :, None] - cB[:, :, None, :] + direct[None])
        flat = dE.reshape(e - s, L * 16)
        flat[:, invalid] = -np.inf
        best[s:e] = flat.max(axis=1)
    return best


def auc(score, label):
    """Rank AUC: P(score of a carrier > score of a non-carrier). 0.5 = no information."""
    label = np.asarray(label, dtype=bool)
    n1, n0 = int(label.sum()), int((~label).sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    order = np.argsort(score, kind='mergesort')
    ranks = np.empty(len(score), dtype=np.float64)
    ranks[order] = np.arange(1, len(score) + 1)
    return (ranks[label].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


print("machinery ready")

In [ ]:
import csv

# =============================================================================
# What to build, and the per-sequence table every figure below is drawn from.
# -----------------------------------------------------------------------------
# PROTEIN picks the alignment; the pair list is the ms0_5 top-1000 ddE list filtered by
# double-mutation-carrier count, the same rule DMC_sebset_freq2.7.1 uses, so the pairs
# here are the pairs in the observed-vs-expected figure.
#
# For every pair we keep the FULL per-sequence arrays, not the per-category aggregates:
# each sequence of the alignment is one background, and dE1/dE2/dE double/p(DMC) are what that
# background does to this pair. That is the whole point of the notebook -- the aggregates
# in the probability CSVs average the background away.
# =============================================================================
PROTEIN = 'RT'            # 'IN' | 'PR' | 'RT'
DMC_MIN_PERCENT = 5.0     # keep a pair when its double-mutation carriers exceed this % of the MSA
TOP_N_PAIRS = 20          # of the survivors, the highest-ddE ones; None -> all

# Gain of fitness = dE double beats the wild type and both single mutants. Set NON_OVERLAPPING
# to True to also require it to beat every double mutation in conflict with the pair (the
# contender rule of DMC_sebset_freq2.7.1). It changes nothing but the four categories, so only
# figure G moves -- and it is the entire cost of this cell, ~35 s on RT against ~1 s.
NON_OVERLAPPING = False

CFG = {
    'IN': (IN_all_seq, IN_consensus_seq, IN_redux, IN_J, IN_weights, 1, 263,
           'IN_top_1000_dde_summary.csv', 'INSTI'),
    'PR': (PR_all_seq, PR_consensus_seq, PR_redux, PR_J, PR_weights, 1, 99,
           'PR_top_1000_dde_summary.csv', 'PI'),
    # NRTI and NNRTI list the same 1000 pairs with the same counts -- RT as a whole
    'RT': (RT_all_seq, RT_consensus_seq, RT_redux, RT_J, RT_weights, 39, 226,
           'RT_NRTI_top_1000_dde_summary.csv', 'NRTI/NNRTI'),
}
ALL_SEQ, CONSENSUS, REDUX, J, WEIGHTS, MIN_POS, MAX_POS, DDE_FILE, DRUG = CFG[PROTEIN]
N_SEQ = len(ALL_SEQ)

with open(data_root / DDE_FILE, newline='') as f:
    _rows = list(csv.DictReader(f))
_cut = N_SEQ * DMC_MIN_PERCENT / 100.0
PAIR_NAMES = [r['double_mutation_pair'] for r in _rows if float(r['both_count']) > _cut]
if TOP_N_PAIRS is not None:
    PAIR_NAMES = PAIR_NAMES[:TOP_N_PAIRS]
GOF_RULE = ('beats the wild type, both single mutants and every contender'
            if NON_OVERLAPPING else 'beats the wild type and both single mutants')
print(f"{PROTEIN}: {len(PAIR_NAMES)} pairs, {N_SEQ} sequences, positions {MIN_POS}-{MAX_POS}")
print(f"gain of fitness: {GOF_RULE}")

CODES, ONEHOT = encode_seqs(ALL_SEQ, MIN_POS, MAX_POS)
_cons_codes, CONS_ONEHOT = encode_seqs([CONSENSUS], MIN_POS, MAX_POS)
U = _cons_codes[0]                                   # consensus residue at every position
T_BG = background_energies(ONEHOT, J) if NON_OVERLAPPING else None
IS_MUT = CODES != U[None, :]                         # per position: does this sequence differ
                                                     # from the consensus?

CATS = ['gain of fitness', 'rescue', 'compensatory', 'non-compensatory']

PAIRS = {}
_competing = {}
for _name in PAIR_NAMES:
    a, b = functions.split_pairs(_name)
    wt1, pos1, mt1 = functions.split_pair(functions.unreduced_to_reduced(REDUX, a))
    wt2, pos2, mt2 = functions.split_pair(functions.unreduced_to_reduced(REDUX, b))
    p1, p2 = pos1 - MIN_POS, pos2 - MIN_POS
    iwt1, imt1 = "ABCD".index(wt1), "ABCD".index(mt1)
    iwt2, imt2 = "ABCD".index(wt2), "ABCD".index(mt2)
    idx = (p1, p2, iwt1, imt1, iwt2, imt2)

    de1, de2, de12, p_dmc = pair_energies(ONEHOT, J, *idx)
    _, _, cons_de12, _ = pair_energies(CONS_ONEHOT, J, *idx)
    dmc = (CODES[:, p1] == imt1) & (CODES[:, p2] == imt2)

    # the four epistasis categories; the contender comparison only runs when asked for
    if NON_OVERLAPPING:
        for key in ((p1, iwt1, imt1), (p2, iwt2, imt2)):
            if key not in _competing:
                _competing[key] = best_competing_dmc(T_BG, J, CODES, key[0], key[1], key[2], U)
        own = de12_from_T(T_BG, J, CODES, p1, p2, imt1, imt2, iwt1, iwt2)
        no_rival = (own > _competing[(p1, iwt1, imt1)]) & (own > _competing[(p2, iwt2, imt2)])
    else:
        no_rival = np.ones(N_SEQ, dtype=bool)

    both_below = (de1 < de12) & (de2 < de12)
    gof = both_below & (de12 > 0) & no_rival
    rescue = both_below & ~gof
    comp = ~both_below & ((de1 < de12) | (de2 < de12))
    cat = np.full(N_SEQ, 3, dtype=np.int8)            # 3 = non-compensatory
    cat[comp] = 2
    cat[rescue] = 1
    cat[gof] = 0

    # background mutational load: how far this sequence is from the consensus, NOT counting
    # the pair's own two positions (otherwise a carrier scores +2 by definition)
    load = IS_MUT.sum(axis=1) - IS_MUT[:, p1] - IS_MUT[:, p2]

    PAIRS[_name] = dict(p1=p1, p2=p2, pos1=pos1, pos2=pos2, idx=idx,
                        de1=de1, de2=de2, de12=de12, p=p_dmc, dmc=dmc, cat=cat, load=load,
                        cons_de12=float(cons_de12[0]),
                        auc_de12=auc(de12, dmc), auc_p=auc(p_dmc, dmc))
    print(f"  {_name:16} carriers {int(dmc.sum()):6d}  "
          f"AUC(dE double) {PAIRS[_name]['auc_de12']:.3f}  gof seqs {int(gof.sum()):6d}")

print(f"\nbuilt per-sequence arrays for {len(PAIRS)} pairs "
      f"({len(PAIRS) * N_SEQ:,} pair x background rows)")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D

# =============================================================================
# Plot style: one validated palette for the whole notebook.
#   - two/three categorical hues (blue, orange, aqua) for series that carry identity
#   - a single blue ramp, light -> dark, where the four epistasis categories are ORDERED
#   - blue <-> red with a gray midpoint where the value is signed (the coupling heatmap)
#   - a de-emphasis gray for "all the other pairs" context lines: the emphasis pattern
#     is what keeps a 20-series figure readable instead of 20 competing colours
# Text always wears ink colours, never a series colour.
# =============================================================================
SURFACE, INK, INK2, MUTED = '#fcfcfb', '#0b0b0b', '#52514e', '#898781'
GRID, BASE, CONTEXT = '#e1e0d9', '#c3c2b7', '#cfcec8'
S1, S2, S3 = '#2a78d6', '#eb6834', '#1baf7a'          # categorical slots 1-3
ORD4 = ['#104281', '#256abf', '#5598e7', '#86b6ef']   # ordered: gof -> non-compensatory
DIVERGE = plt.get_cmap('RdBu_r')                       # signed values, gray-ish midpoint

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.edgecolor': BASE, 'axes.labelcolor': INK2,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK,
    'grid.color': GRID, 'grid.linewidth': 0.8, 'legend.frameon': False,
})


def style(ax, xlabel=None, ylabel=None, title=None, grid='y'):
    """Recessive chrome: no top/right spines, hairline grid, muted ticks."""
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(BASE)
    ax.tick_params(length=0, labelsize=9)
    if grid:
        ax.grid(axis=grid, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    if xlabel:
        ax.set_xlabel(xlabel, color=INK2)
    if ylabel:
        ax.set_ylabel(ylabel, color=INK2)
    if title:
        ax.set_title(title, color=INK, loc='left', pad=10)
    return ax


def header(ax, title, note=None):
    """Title above the axes, with an optional note line under it.

    Placed in inches rather than axes fractions so the two lines keep their spacing
    whatever the figure height -- set_title plus a second text at a fixed fraction
    collide as soon as a figure gets tall.
    """
    h = ax.get_position().height * ax.figure.get_figheight()
    y0, line = 0.06 / h, 0.26 / h
    if note:
        ax.text(0, 1 + y0, note, transform=ax.transAxes, fontsize=9.5, color=INK2, va='bottom')
        ax.text(0, 1 + y0 + line, title, transform=ax.transAxes, fontsize=11.5, color=INK,
                va='bottom')
    else:
        ax.text(0, 1 + y0, title, transform=ax.transAxes, fontsize=11.5, color=INK, va='bottom')


def save(fig, name):
    path = f"background_{name}_{PROTEIN}.png"
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor=SURFACE)
    print("saved", path)


def decile_curve(x, carrier, bins=10):
    """Carrier fraction against x, in equal-count bins of x. Returns (mean x, fraction, n)."""
    edges = np.quantile(x, np.linspace(0, 1, bins + 1))
    edges[-1] += 1e-9
    which = np.clip(np.searchsorted(edges, x, side='right') - 1, 0, bins - 1)
    mx, fr, n = [], [], []
    for k in range(bins):
        m = which == k
        if m.sum():
            mx.append(x[m].mean()); fr.append(carrier[m].mean()); n.append(int(m.sum()))
    return np.array(mx), np.array(fr), np.array(n)


# the three pairs used as named examples throughout: best / median / worst discrimination
_by_auc = sorted(PAIRS, key=lambda k: PAIRS[k]['auc_de12'])
EXAMPLES = [_by_auc[-1], _by_auc[len(_by_auc) // 2], _by_auc[0]][:3]
EX_COLOR = dict(zip(EXAMPLES, [S1, S2, S3]))
print("example pairs (strongest / median / weakest AUC):", EXAMPLES)

In [ ]:
# =============================================================================
# FIGURE A -- the consensus background is not the background
#
# The field usually reports ONE dE double (or ddE) per pair, computed on a reference sequence.
# This figure shows what that number hides. For every pair:
#     grey line   = the 5th-95th percentile of dE double across all backgrounds in the alignment
#     diamond     = dE double on the consensus background -- the single number one would report
#     orange dot  = mean dE double over the sequences that do NOT carry the pair
#     blue dot    = mean dE double over the sequences that DO carry it
#
# Read it against the dashed line at dE double = 0, which is where the double mutant stops being
# worse than the wild type. The consensus diamonds sit on the negative side for essentially
# every pair -- on the reference sequence these mutations are unfavourable -- while the blue
# carrier means sit on the positive side. The pair did not change; the background did.
# =============================================================================
order = sorted(PAIRS, key=lambda k: PAIRS[k]['de12'][PAIRS[k]['dmc']].mean()
               if PAIRS[k]['dmc'].any() else -99)
y = np.arange(len(order))

fig, ax = plt.subplots(figsize=(9.5, 0.42 * len(order) + 2.2))
for i, name in enumerate(order):
    d = PAIRS[name]
    lo, hi = np.percentile(d['de12'], [5, 95])
    ax.plot([lo, hi], [i, i], color=CONTEXT, linewidth=3, solid_capstyle='round', zorder=1)

ax.axvline(0, color=BASE, linestyle='--', linewidth=1, zorder=0)
carrier_mean = [PAIRS[n]['de12'][PAIRS[n]['dmc']].mean() for n in order]
noncarr_mean = [PAIRS[n]['de12'][~PAIRS[n]['dmc']].mean() for n in order]
cons = [PAIRS[n]['cons_de12'] for n in order]

ax.scatter(noncarr_mean, y, s=55, color=S2, edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.scatter(carrier_mean, y, s=55, color=S1, edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.scatter(cons, y, s=70, marker='D', facecolor=SURFACE, edgecolor=INK, linewidth=1.4, zorder=4)

ax.set_yticks(y, order, fontsize=9, color=INK2)
ax.set_ylim(-0.8, len(order) - 0.2)
style(ax, xlabel='dE double  (negative = the double mutant is worse than the wild type)', grid='x')
ax.legend(handles=[
    Line2D([], [], color=CONTEXT, lw=3, label='5th-95th percentile across backgrounds'),
    Line2D([], [], marker='D', color=INK, markerfacecolor=SURFACE, lw=0, markersize=8,
           label='consensus background (the single reported number)'),
    Line2D([], [], marker='o', color=S2, lw=0, markersize=8, label='mean over non-carriers'),
    Line2D([], [], marker='o', color=S1, lw=0, markersize=8, label='mean over carriers'),
], loc='upper left', bbox_to_anchor=(0.0, -0.045), ncol=2, fontsize=9, labelcolor=INK2)

n_neg = sum(c < 0 for c in cons)
n_pos = sum(c > 0 for c in carrier_mean)
header(ax, f'{DRUG}: the same pair, scored on every background in the alignment',
       f'{n_neg} of {len(order)} pairs are unfavourable on the consensus background; '
       f'{n_pos} of {len(order)} are favourable in the backgrounds that actually carry them')
save(fig, 'A_consensus_vs_realized')
plt.show()

In [ ]:
# =============================================================================
# FIGURE B -- how well the background alone decides who carries the pair
#
# Treat it as a prediction problem: score every sequence by what the background does to the
# pair (dE double, or p(DMC)), and ask whether the sequences that actually carry the double
# mutation score higher. AUC is the probability that a randomly chosen carrier outscores a
# randomly chosen non-carrier -- 0.5 is no information, 1.0 is perfect separation.
#
#   left   ROC curves for three pairs: the strongest, the median and the weakest.
#   right  one row per pair, the two scores joined: dE double (orange) and p(DMC) (blue).
#
# The weakest pair is the informative one, not an embarrassment: it is the pair whose dE double
# barely moves across backgrounds (see figure A), and where the background does not modulate
# the pair, it also does not predict it. That is the control the story needs.
# =============================================================================
fig, ax = plt.subplots(figsize=(10, 0.34 * len(PAIRS) + 2.8))

order = sorted(PAIRS, key=lambda k: PAIRS[k]['auc_de12'])
y = np.arange(len(order))
a_de = np.array([PAIRS[n]['auc_de12'] for n in order])
a_p = np.array([PAIRS[n]['auc_p'] for n in order])
for i, (x0, x1) in enumerate(zip(a_de, a_p)):
    ax.plot([x0, x1], [i, i], color=CONTEXT, linewidth=2.5, solid_capstyle='round', zorder=1)
ax.scatter(a_de, y, s=55, color=S2, edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.scatter(a_p, y, s=55, color=S1, edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.axvline(0.5, color=BASE, linestyle='--', linewidth=1, zorder=0)
ax.text(0.502, -0.5, 'no information', color=MUTED, fontsize=9, va='center')
ax.set_yticks(y, order, fontsize=9, color=INK2)
ax.set_ylim(-1.0, len(order) - 0.2)
ax.set_xlim(0.45, 1.005)
style(ax, xlabel='AUC: P(a carrier scores above a non-carrier)', grid='x')
ax.legend(handles=[Line2D([], [], marker='o', color=S2, lw=0, markersize=8, label='dE double'),
                   Line2D([], [], marker='o', color=S1, lw=0, markersize=8, label='p(DMC)')],
          loc='lower right', fontsize=9, labelcolor=INK2)
header(ax, f'{DRUG}: how well the background alone decides who carries the pair',
       f'median AUC(dE double) {np.median(a_de):.3f}   median AUC(p) {np.median(a_p):.3f}   '
       f'(inset: the same ranking drawn as ROC curves, strongest / median / weakest pair)')

# the ROC curves live in the empty upper-left of the dumbbell panel
axins = ax.inset_axes([0.05, 0.46, 0.33, 0.46])
for name in EXAMPLES:
    d = PAIRS[name]
    o = np.argsort(-d['de12'])
    tp = np.cumsum(d['dmc'][o]) / max(d['dmc'].sum(), 1)
    fp = np.cumsum(~d['dmc'][o]) / max((~d['dmc']).sum(), 1)
    axins.plot(np.r_[0, fp], np.r_[0, tp], color=EX_COLOR[name], linewidth=2, zorder=3,
               label=f"{name}  {d['auc_de12']:.3f}")
axins.plot([0, 1], [0, 1], color=BASE, linestyle='--', linewidth=1, zorder=1)
axins.set_aspect('equal')
axins.set_xlim(0, 1)
axins.set_ylim(0, 1)
style(axins, xlabel='false positive rate', ylabel='carriers recovered', grid='both')
axins.xaxis.label.set_size(8.5)
axins.yaxis.label.set_size(8.5)
axins.tick_params(labelsize=8)
axins.legend(loc='lower right', fontsize=8, labelcolor=INK2)

save(fig, 'B_auc')
plt.show()

In [ ]:
# =============================================================================
# FIGURE C -- the dose-response: more favourable background, more carriers
#
# For each pair, the alignment is split into ten equal-sized bins of dE double and the fraction
# of sequences carrying the double mutation is plotted per bin. If the background decides,
# these curves must rise -- and they must rise for every pair, not just the famous ones.
#
#   left   raw dE double on the x axis, so pairs sit where their energies actually are
#   right  the same curves against the WITHIN-PAIR decile index, which removes each pair's
#          own offset; if the curves collapse onto one shape, the model is measuring the
#          same thing in every pair rather than something pair-specific
#
# Twenty coloured lines would be unreadable and would also imply the pairs are the subject.
# They are not -- the shared shape is. So all pairs are drawn in grey, three are named, and
# the thick dark line is the pooled curve over every pair x background row.
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

for ax, mode in zip(axes, ('raw', 'decile')):
    for name, d in PAIRS.items():
        mx, fr, _ = decile_curve(d['de12'], d['dmc'])
        x = mx if mode == 'raw' else np.arange(1, len(fr) + 1)
        if name in EX_COLOR:
            continue
        ax.plot(x, fr, color=CONTEXT, linewidth=1.2, zorder=2)
    ends = []
    for name in EXAMPLES:
        d = PAIRS[name]
        mx, fr, _ = decile_curve(d['de12'], d['dmc'])
        x = mx if mode == 'raw' else np.arange(1, len(fr) + 1)
        ax.plot(x, fr, color=EX_COLOR[name], linewidth=2.2, zorder=4)
        ends.append([x[-1], fr[-1], name])

    # direct labels at the end of each curve, nudged apart when two curves finish at the
    # same height -- M41L-T215Y and M41L-L210W both top out near 95% of backgrounds
    ends.sort(key=lambda e: e[1])
    for k in range(1, len(ends)):
        ends[k][1] = max(ends[k][1], ends[k - 1][1] + 0.055)
    for xe, ye, name in ends:
        ax.annotate(name, (xe, ye), textcoords='offset points', xytext=(6, 0),
                    fontsize=9, color=INK2, va='center')

    if mode == 'decile':
        pooled = np.zeros(10)
        for k in range(10):
            num = den = 0
            for d in PAIRS.values():
                q = np.quantile(d['de12'], [k / 10, (k + 1) / 10])
                m = (d['de12'] >= q[0]) & (d['de12'] <= q[1])
                num += d['dmc'][m].sum(); den += m.sum()
            pooled[k] = num / den
        ax.plot(np.arange(1, 11), pooled, color=INK, linewidth=2.6, zorder=5)
        ax.annotate('pooled over all pairs', (10, pooled[-1]), textcoords='offset points',
                    xytext=(6, 0), fontsize=9, color=INK, va='center')
        style(ax, xlabel='within-pair dE double decile (1 = least favourable background)',
              ylabel='fraction carrying the pair', title='each pair on its own scale')
        ax.set_xticks(range(1, 11))
    else:
        style(ax, xlabel='dE double (bin mean)', ylabel='fraction carrying the pair',
              title=f'{DRUG}: carriers against what the background does')

fig.tight_layout()
save(fig, 'C_dose_response')
plt.show()

In [ ]:
# =============================================================================
# FIGURE D -- is the per-background probability calibrated?
#
# p(DMC) is a probability, so it can be checked the way any probability is: bin the
# backgrounds by what the model predicted, and plot what actually happened. A calibrated
# model puts the points on the diagonal -- among backgrounds where it says 30%, about 30%
# of sequences carry the pair.
#
# This is the observed-vs-expected figure at per-sequence resolution. The four-panel version
# compares one aggregate number per pair per epistasis category; here every point is a bin of
# individual backgrounds, so it tests the model across the whole continuum instead of at four
# coarse summaries. The bins hold equal numbers of backgrounds, so most of them sit at low
# p -- that is where most backgrounds are -- and the markers crowd near the origin.
# =============================================================================
fig, ax = plt.subplots(figsize=(7.4, 7.0))

for name, d in PAIRS.items():
    mx, fr, _ = decile_curve(d['p'], d['dmc'])
    ax.plot(mx, fr, color=CONTEXT, linewidth=1.1, zorder=2)

p_all = np.concatenate([d['p'] for d in PAIRS.values()])
c_all = np.concatenate([d['dmc'] for d in PAIRS.values()])
mx, fr, n = decile_curve(p_all, c_all, bins=20)
ax.plot([0, 1], [0, 1], color=BASE, linestyle='--', linewidth=1, zorder=1)
ax.plot(mx, fr, color=S1, linewidth=2, zorder=4)
# equal-count bins, so marker size would encode nothing -- keep it constant
ax.scatter(mx, fr, s=70, color=S1, edgecolor=SURFACE, linewidth=1.2, zorder=5)

ax.set_aspect('equal')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
style(ax, xlabel='predicted p(DMC) for the background (bin mean)',
      ylabel='fraction of those backgrounds carrying the pair',
      title=f'{DRUG}: calibration across {len(PAIRS) * N_SEQ:,} pair x background rows',
      grid='both')
ax.legend(handles=[
    Line2D([], [], color=S1, lw=2, marker='o', markersize=8, label='pooled, 20 equal-count bins'),
    Line2D([], [], color=CONTEXT, lw=1.5, label='one pair, 10 bins'),
    Line2D([], [], color=BASE, lw=1, linestyle='--', label='perfect calibration'),
], loc='upper left', fontsize=9, labelcolor=INK2)
save(fig, 'D_calibration')
plt.show()

In [ ]:
# =============================================================================
# FIGURE E -- background load, and the control that rules out the boring explanation
#
# The simplest summary of a background is how far it is from the consensus, counting every
# position except the pair's own two. Carriers concentrate in the more mutated backgrounds
# (left panel) -- which invites the objection that heavily mutated sequences simply contain
# more of everything, and that no coupling is needed to explain it.
#
# The right panel is the answer to that objection. Within each load stratum -- comparing only
# sequences that carry a similar number of other mutations -- dE double still separates carriers
# from non-carriers, with AUC staying far above the 0.5 no-information line. It is not the
# count of other mutations that decides, it is which ones they are and how they couple.
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

ax = axes[0]
for name, d in PAIRS.items():
    mx, fr, _ = decile_curve(d['load'].astype(float), d['dmc'])
    if name in EX_COLOR:
        continue
    ax.plot(mx, fr, color=CONTEXT, linewidth=1.2, zorder=2)
for name in EXAMPLES:
    d = PAIRS[name]
    mx, fr, _ = decile_curve(d['load'].astype(float), d['dmc'])
    ax.plot(mx, fr, color=EX_COLOR[name], linewidth=2.2, zorder=4)
    ax.annotate(name, (mx[-1], fr[-1]), textcoords='offset points', xytext=(6, 0),
                fontsize=9, color=INK2, va='center')
style(ax, xlabel='background mutational load (positions differing from consensus,\n'
                 "excluding the pair's own two)",
      ylabel='fraction carrying the pair',
      title=f'{DRUG}: carriers concentrate in mutated backgrounds')

ax = axes[1]
load0 = next(iter(PAIRS.values()))['load']
edges = np.quantile(load0, [0, 0.25, 0.5, 0.75, 1.0])
centres = np.arange(4)
for name, d in PAIRS.items():
    vals = []
    for k in range(4):
        m = ((d['load'] >= edges[k]) & (d['load'] <= edges[k + 1]) if k == 3
             else (d['load'] >= edges[k]) & (d['load'] < edges[k + 1]))
        vals.append(auc(d['de12'][m], d['dmc'][m]) if 0 < d['dmc'][m].sum() < m.sum() else np.nan)
    col = EX_COLOR.get(name, CONTEXT)
    ax.plot(centres, vals, color=col, linewidth=2.2 if name in EX_COLOR else 1.2,
            zorder=4 if name in EX_COLOR else 2)
    if name in EX_COLOR:
        ax.annotate(name, (centres[-1], vals[-1]), textcoords='offset points', xytext=(6, 0),
                    fontsize=9, color=INK2, va='center')
ax.axhline(0.5, color=BASE, linestyle='--', linewidth=1, zorder=0)
ax.text(0, 0.515, 'no information', color=MUTED, fontsize=9)
ax.set_xticks(centres, [f'Q{k+1}\n{int(edges[k])}-{int(edges[k+1])} mutations' for k in range(4)],
              fontsize=9)
ax.set_ylim(0.45, 1.02)
style(ax, ylabel='AUC of dE double within the stratum',
      title='the same test, inside each load stratum')

fig.tight_layout()
save(fig, 'E_background_load')
plt.show()

In [ ]:
# =============================================================================
# FIGURE F -- WHICH positions in the background do the work
#
# dE double splits exactly into one term per background position: for a position q carrying
# residue s_q, its contribution to this pair is
#
#     c(q) = [J(p1,q,wt1,s_q) - J(p1,q,mt1,s_q)] + [J(p2,q,wt2,s_q) - J(p2,q,mt2,s_q)]
#
# and summing c(q) over every q outside the pair, plus the direct p1-p2 coupling, returns
# dE double exactly -- asserted below, so the decomposition is checked rather than assumed. The
# background is therefore not a black box: it is a list of named positions, each pushing the
# pair up or down.
#
# What is plotted is the LEVER at each position: how much dE double moves when q carries a
# mutation instead of the consensus residue,
#
#     lever(q) = mean over sequences mutated at q of [ c(q) - c(q at consensus) ],
#
# which is the quantity a reader cares about -- "if this accessory mutation is present, does
# the pair become more likely?" -- rather than the raw level, which mostly reports how often
# q is mutated at all. Red = a mutation at that position pushes the pair towards happening.
# The pair's own two positions carry no lever by construction (their coupling is the direct
# term, not a background term) and are left out, as are positions mutated in fewer than
# MIN_MUTATED sequences, where the mean would be noise.
#
# Rows are pairs, columns the positions with the strongest levers overall. Columns that light
# up across many rows are the accessory sites the whole resistance pathway leans on.
# =============================================================================
TOP_POSITIONS = 25
MIN_MUTATED = 20

L = MAX_POS - MIN_POS + 1
qs = np.arange(L)
names = list(PAIRS)
lever = np.full((len(names), L), np.nan)

for i, name in enumerate(names):
    d = PAIRS[name]
    p1, p2, iwt1, imt1, iwt2, imt2 = d['idx']
    # c(q) for every sequence and every position, and the same at the consensus residue
    c = ((J[p1][qs[None, :], iwt1, CODES] - J[p1][qs[None, :], imt1, CODES])
         + (J[p2][qs[None, :], iwt2, CODES] - J[p2][qs[None, :], imt2, CODES]))
    c[:, p1] = 0.0
    c[:, p2] = 0.0
    direct = J[p1, p2, iwt1, iwt2] - J[p1, p2, imt1, imt2]
    assert np.allclose(c.sum(axis=1) + direct, d['de12'], atol=1e-3), name   # exact split

    c_cons = ((J[p1][qs, iwt1, U] - J[p1][qs, imt1, U])
              + (J[p2][qs, iwt2, U] - J[p2][qs, imt2, U]))
    c_cons[p1] = c_cons[p2] = 0.0
    for q in range(L):
        if q in (p1, p2):
            continue
        m = IS_MUT[:, q]
        if m.sum() >= MIN_MUTATED:
            lever[i, q] = (c[m, q] - c_cons[q]).mean()

score = np.nanmean(np.abs(lever), axis=0)
top = np.argsort(-np.nan_to_num(score))[:TOP_POSITIONS]
top = top[np.argsort(top)]
M = lever[:, top]
lim = np.nanmax(np.abs(M))

fig, ax = plt.subplots(figsize=(0.46 * TOP_POSITIONS + 3.6, 0.38 * len(names) + 2.6))
im = ax.imshow(M, cmap=DIVERGE, norm=TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim),
               aspect='auto')
ax.set_xticks(np.arange(TOP_POSITIONS), [str(q + MIN_POS) for q in top], fontsize=8.5,
              rotation=90)
ax.set_yticks(np.arange(len(names)), names, fontsize=9, color=INK2)
ax.set_xticks(np.arange(-0.5, TOP_POSITIONS, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(names), 1), minor=True)
ax.grid(which='minor', color=SURFACE, linewidth=2)   # 2px surface gap between cells
ax.tick_params(which='minor', length=0)
ax.tick_params(length=0)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_xlabel('background position', color=INK2)
header(ax, f'{DRUG}: what a mutation at each background position does to the pair',
       'blank = the pair\'s own position, or fewer than '
       f'{MIN_MUTATED} sequences mutated there')
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cb.set_label('lever on dE double:  red = a mutation here makes the pair more likely',
             color=INK2, fontsize=9)
cb.ax.tick_params(colors=MUTED, labelsize=8.5, length=0)
cb.outline.set_visible(False)
save(fig, 'F_position_levers')
plt.show()

In [ ]:
# =============================================================================
# FIGURE G -- the background walks the pair through the categories
#
# Same load axis as figure E, but instead of asking how many sequences carry the pair, this
# asks what the model calls the background. Every sequence falls in exactly one of the four
# epistasis categories, so the bars are a part-to-whole and stack to 100%.
#
# The categories are ORDERED (gain of fitness > rescue > compensatory > non-compensatory),
# so they take one blue ramp light-to-dark rather than four unrelated hues -- the shade
# itself carries the ranking. Reading left to right: in near-consensus backgrounds almost
# everything is non-compensatory, and gain of fitness only appears once enough of the rest
# of the protein has moved. Gain of fitness is a minority-background phenomenon, which is
# the argument for scoring every sequence instead of quoting one number per pair.
#
# This is the only figure the NON_OVERLAPPING flag touches -- the rule in force is printed
# under the title, since it decides where the gain-of-fitness / rescue line falls.
# =============================================================================
bins = 8
load0 = next(iter(PAIRS.values()))['load'].astype(float)
edges = np.quantile(load0, np.linspace(0, 1, bins + 1))
edges[-1] += 1e-9

frac = np.zeros((bins, 4))
counts = np.zeros(bins)
for k in range(bins):
    tot = np.zeros(4)
    for d in PAIRS.values():
        m = (d['load'] >= edges[k]) & (d['load'] < edges[k + 1])
        tot += np.bincount(d['cat'][m], minlength=4)
    counts[k] = tot.sum()
    frac[k] = tot / tot.sum()

fig, ax = plt.subplots(figsize=(10.5, 5.6))
bottom = np.zeros(bins)
x = np.arange(bins)
for c in range(4):
    ax.bar(x, frac[:, c], bottom=bottom, width=0.78, color=ORD4[c],
           edgecolor=SURFACE, linewidth=2, zorder=3)
    # direct labels: mandatory with four stacked series, and they double as the legend
    for k in range(bins):
        if frac[k, c] > 0.055:
            ax.text(x[k], bottom[k] + frac[k, c] / 2, f'{100*frac[k, c]:.0f}%',
                    ha='center', va='center', fontsize=8.5,
                    color=SURFACE if c < 2 else INK, zorder=4)
    ax.annotate(CATS[c], (bins - 0.55, bottom[-1] + frac[-1, c] / 2),
                textcoords='offset points', xytext=(8, 0), fontsize=9.5, color=INK2,
                va='center')
    bottom += frac[:, c]

ax.set_xticks(x, [f'{int(edges[k])}-{int(edges[k+1])}' for k in range(bins)], fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks(np.linspace(0, 1, 6), [f'{int(v*100)}%' for v in np.linspace(0, 1, 6)])
style(ax, xlabel="background mutational load (excluding the pair's own two positions)",
      ylabel='share of pair x background rows')
header(ax, f'{DRUG}: what the model calls the background, by how mutated it is',
       f'gain of fitness = {GOF_RULE}')
ax.set_xlim(-0.6, bins + 1.6)
save(fig, 'G_category_by_load')
plt.show()